# Caso práctico: capacidad de red de AndesTel

## Contexto empresarial

AndesTel Telecom S.A.C. brinda servicios de Internet residencial mediante redes FTTH y HFC en distintas zonas de Lima y Callao.

El área comercial quiere lanzar una campaña para captar nuevos clientes. Sin embargo, el área de Operaciones advierte que algunas zonas podrían encontrarse cerca del límite de capacidad durante las horas de mayor tráfico.

La gerencia solicita al equipo de Analítica determinar:

**¿En qué zonas existe capacidad suficiente para continuar vendiendo nuevos servicios y en qué zonas es necesario ampliar la infraestructura antes de captar más clientes?**

## Dataset

Zonas: Representa la infraestructura disponible (zonas.csv).

| Campo              | Descripción                   |
| ------------------ | ----------------------------- |
| `zona_id`          | Código de zona                |
| `zona`             | Nombre                        |
| `region`           | Sector geográfico             |
| `tecnologia`       | FTTH / HFC                    |
| `capacidad_mbps`   | Capacidad total del enlace    |
| `limite_operacion` | Porcentaje máximo recomendado |

Clientes: Contiene el parque comercial (clientes.csv).

| Campo        | Descripción               |
| ------------ | ------------------------- |
| `cliente_id` | Identificador             |
| `zona_id`    | Zona donde está conectado |
| `plan_mbps`  | Velocidad contratada      |
| `estado`     | ACTIVO / SUSPENDIDO       |
| `fecha_alta` | Fecha de alta             |


Trafico de red: Contendrá mediciones horarias durante un mes (trafico_red.csv).

| Campo                  | Descripción       |
| ---------------------- | ----------------- |
| `timestamp`            | Fecha y hora      |
| `zona_id`              | Zona              |
| `trafico_down_mbps`    | Tráfico download  |
| `trafico_up_mbps`      | Tráfico upload    |
| `latencia_ms`          | Latencia promedio |
| `perdida_paquetes_pct` | Packet loss       |


## Supuesto

### Política de capacidad de AndesTel

Para efectos del caso, la empresa establece que no debe planificar ventas utilizando el 100% de la capacidad física. El máximo recomendado será: **80%**

| Utilización P95 | Estado        | Decisión                          |
| --------------: | ------------- | --------------------------------- |
|           ≤ 65% | 🟢 Saludable  | Se pueden captar nuevos clientes  |
|     >65% y ≤80% | 🟡 Preventivo | Venta controlada                  |
|            >80% | 🔴 Crítico    | No vender hasta ampliar capacidad |


Para el ejercicio utilizaremos el percentil 95 del tráfico horario:

**El P95 representa un nivel de tráfico elevado que se alcanza o supera aproximadamente durante el 5% de las mediciones.**

En una operación real dependería de la arquitectura, redundancia, SLA, tecnología, tráfico upstream/downstream y criterios de ingeniería del operador.

In [ ]:
## 1. Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
## 2. Cargar los datos del archivo CSV
df_clientes = pd.read_csv('./data/caso1_clientes.csv')
df_zonas = pd.read_csv('./data/caso1_zonas.csv')
df_trafico = pd.read_csv('./data/caso1_trafico_red.csv')

## Mostrar las primeras filas de cada DataFrame
print("Datos de Clientes:")
display(df_clientes.head())

print("\nDatos de Zonas:")
display(df_zonas.head())

print("\nDatos de Tráfico de Red:")
display(df_trafico.head())

In [ ]:
## 3. Visualizar el tamaño de los dataframes
print(f"Tamaño del DataFrame de Clientes: {df_clientes.shape}")
print(f"Tamaño del DataFrame de Zonas: {df_zonas.shape}")
print(f"Tamaño del DataFrame de Tráfico de Red: {df_trafico.shape}")

In [ ]:
## 4. Verificando valores nulos
print("Valores nulos en Clientes:")
print(df_clientes.isnull().sum().sort_values(ascending=False))

print("\nValores nulos en Zonas:")
print(df_zonas.isnull().sum().sort_values(ascending=False))

print("\nValores nulos en Tráfico de Red:")
print(df_trafico.isnull().sum().sort_values(ascending=False))

In [ ]:
## 5. Eliminar nulos de los DataFrames
df_clientes = df_clientes.dropna()
df_zonas = df_zonas.dropna()
df_trafico = df_trafico.dropna()

## Volver a mostrar el tamaño de los dataframes después de eliminar nulos
print("\nDespués de eliminar nulos:")
print(f"Tamaño del DataFrame de Clientes: {df_clientes.shape}")
print(f"Tamaño del DataFrame de Zonas: {df_zonas.shape}")
print(f"Tamaño del DataFrame de Tráfico de Red: {df_trafico.shape}")

In [ ]:
## 6. Revisar datos duplicados en los DataFrames
print("\nDatos duplicados en Clientes:")
print(df_clientes[df_clientes.duplicated()])
print("\nDatos duplicados en Zonas:")
print(df_zonas[df_zonas.duplicated()])
print("\nDatos duplicados en Tráfico de Red:")
print(df_trafico[df_trafico.duplicated()])

In [ ]:
## 7. Eliminar duplicados de los DataFrames
df_clientes = df_clientes.drop_duplicates()
df_zonas = df_zonas.drop_duplicates()
df_trafico = df_trafico.drop_duplicates()

## Volver a mostrar el tamaño de los dataframes después de eliminar duplicados
print("\nDespués de eliminar duplicados:")
print(f"Tamaño del DataFrame de Clientes: {df_clientes.shape}")
print(f"Tamaño del DataFrame de Zonas: {df_zonas.shape}")
print(f"Tamaño del DataFrame de Tráfico de Red: {df_trafico.shape}")

In [ ]:
## 8. Normalizar todos los datos categóricos a mayusculas en todos los dataframes
categorical_columns_clientes = df_clientes.select_dtypes(include=['object']).columns
categorical_columns_zonas = df_zonas.select_dtypes(include=['object']).columns
categorical_columns_trafico = df_trafico.select_dtypes(include=['object']).columns

df_clientes[categorical_columns_clientes] = df_clientes[categorical_columns_clientes].apply(lambda x: x.str.upper().str.strip())
df_zonas[categorical_columns_zonas] = df_zonas[categorical_columns_zonas].apply(lambda x: x.str.upper().str.strip())
df_trafico[categorical_columns_trafico] = df_trafico[categorical_columns_trafico].apply(lambda x: x.str.upper().str.strip())


In [ ]:
## Mostrar las primeras filas de cada DataFrame despues de normalizar los datos categóricos
print("Datos de Clientes:")
display(df_clientes.head())

print("\nDatos de Zonas:")
display(df_zonas.head())

print("\nDatos de Tráfico de Red:")
display(df_trafico.head())

In [ ]:
## 9. Convertir a fechas los campos feca_alta de clientes y timestamp de trafico_red
df_clientes['fecha_alta'] = pd.to_datetime(df_clientes['fecha_alta'], errors='coerce', format='%Y-%m-%d')
df_trafico['timestamp'] = pd.to_datetime(df_trafico['timestamp'], errors='coerce', format='%Y-%m-%d %H:%M:%S')

In [ ]:
## 10. Eliminar datos incoherentes
df_clientes = df_clientes[df_clientes['plan_mbps'] > 0]
df_zonas = df_zonas[df_zonas['capacidad_mbps'] > 0]
df_zonas = df_zonas[(df_zonas['limite_operacion'] >= 0) & (df_zonas['limite_operacion'] <= 1)]
df_trafico = df_trafico[df_trafico['trafico_down_mbps'] >= 0]
df_trafico = df_trafico[df_trafico['trafico_up_mbps'] >= 0]
df_trafico = df_trafico[df_trafico['latencia_ms'] >= 0]

In [ ]:
## 11. Mostrar cantidad de clientes por zona
clientes_por_zona = df_clientes.groupby('zona_id').size().reset_index(name='cantidad_clientes')
print("\nCantidad de clientes por zona:")
display(clientes_por_zona)

In [ ]:
## 12. Mostrar la cantidad de clientes por zona y plan contrado (pivot table)
clientes_por_zona_y_plan = df_clientes.pivot_table(index='zona_id', columns='plan_mbps', values='cliente_id', aggfunc='count', fill_value=0)
print("\nCantidad de clientes por zona y plan contratado:")
display(clientes_por_zona_y_plan)

In [ ]:
## 13. Crear las columnas fecha y hora del timestamp en el DataFrame de tráfico de red
df_trafico['fecha'] = df_trafico['timestamp'].dt.date
df_trafico['hora'] = df_trafico['timestamp'].dt.time

In [ ]:
## 14. Mostrar el tráfico de red de bajada (trafico_down_mbps) promedio por hora del día
trafico_promedio_por_hora = df_trafico.groupby('hora')['trafico_down_mbps'].mean().reset_index(name='promedio_trafico_down_mbps')
print("\nTráfico de red de bajada promedio por hora del día:")
display(trafico_promedio_por_hora)

In [ ]:
## Graficar el tráfico de red de bajada promedio por hora del día
plt.figure(figsize=(12, 6))
plt.plot(trafico_promedio_por_hora['hora'].astype(str), trafico_promedio_por_hora['promedio_trafico_down_mbps'], marker='o')
plt.xticks(rotation=45)
plt.title('Tráfico de red de bajada promedio por hora del día')
plt.xlabel('Hora')
plt.ylabel('Promedio tráfico down (Mbps)')
plt.grid(True)
plt.show()

In [ ]:
## 15. Mostrar el tráfico de red de bajada (trafico_down_mbps) promedio por hora del día por zona (pivot table)
trafico_promedio_por_hora_y_zona = df_trafico.pivot_table(index='hora', columns='zona_id', values='trafico_down_mbps', aggfunc='mean', fill_value=0)
print("\nTráfico de red de bajada promedio por hora del día por zona:")
display(trafico_promedio_por_hora_y_zona)

In [ ]:
## 16. Mostrar el tráfico de red de bajada (trafico_down_mbps) promedio por hora del día por zona (gráfico)
plt.figure(figsize=(12, 6))
for zona in trafico_promedio_por_hora_y_zona.columns:
    plt.plot(trafico_promedio_por_hora_y_zona.index.astype(str), trafico_promedio_por_hora_y_zona[zona], marker='o', label=f'Zona {zona}')
plt.title('Tráfico de red de bajada promedio por hora del día por zona')
plt.xticks(rotation=45)
plt.xlabel('Hora')
plt.ylabel('Promedio tráfico down (Mbps)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
## 17. Agrupar el trafico de red por zona y calcular el trafico promedio de bajada, el maximo trafico de bajada,
# el percentil 95 del trafico de bajada, la latencia promedio y la perdida de paquetes promedio
trafico_agrupado_por_zona = df_trafico.groupby('zona_id').agg(
    promedio_trafico_down_mbps=('trafico_down_mbps', 'mean'),
    maximo_trafico_down_mbps=('trafico_down_mbps', 'max'),
    percentil_95_trafico_down_mbps=('trafico_down_mbps', lambda x: x.quantile(0.95)),
    promedio_latencia_ms=('latencia_ms', 'mean'),
    promedio_perdida_paquetes=('perdida_paquetes_pct', 'mean')
).reset_index()
display(trafico_agrupado_por_zona)

In [ ]:
## 18. Calcular la capacidad utilizada por zona (trafico p95 / capacidad_mbps)
df_capacidad = trafico_agrupado_por_zona.merge(df_zonas[['zona_id', 'capacidad_mbps']], on='zona_id', how='left')
df_capacidad['capacidad_utilizada_pct'] = (df_capacidad['percentil_95_trafico_down_mbps'] / df_capacidad['capacidad_mbps']) * 100
display(df_capacidad)

In [ ]:
## 19. Mostrar las zonas con su capacidad utilizada y la recomendacion de impulsar nuevos clientes de acuerdo con:
# ≤ 65% => 🟢 Saludable (Se pueden captar nuevos clientes)
# >65% y ≤80% => 🟡 Preventivo (Venta controlada)
# >80% => 🔴 Crítico  (No vender hasta ampliar capacidad)
def recomendar_venta(capacidad_utilizada):
    if capacidad_utilizada <= 65:
        return "🟢 Saludable (Se pueden captar nuevos clientes)"
    elif 65 < capacidad_utilizada <= 80:
        return "🟡 Preventivo (Venta controlada)"
    else:
        return "🔴 Crítico (No vender hasta ampliar capacidad)"

df_capacidad['capacidad_utilizada_pct'] = df_capacidad['capacidad_utilizada_pct'].apply(lambda x: round(x, 2))
df_capacidad['recomendacion_venta'] = df_capacidad['capacidad_utilizada_pct'].apply(recomendar_venta)
display(df_capacidad[['zona_id', 'capacidad_utilizada_pct', 'recomendacion_venta']])